In [11]:
# Imports
import pandas as pd
import numpy as np
import spacy
from dateparser.search import search_dates
from dateutil.parser import ParserError
from dateutil import parser as dateutil_parser
from datetime import datetime
import re
import unicodedata
from collections import defaultdict
from tqdm import tqdm


# Loading the csv into a dataframe and selecting the columns of interest
file_path = r'_github-AAC_accidents_tagged_data.xlsx'
df_long = pd.read_excel(file_path)
df = df_long[['ID', 'Accident Title', 'Publication Year', 'Text', 'Tags Applied']].copy()


In [12]:
# extracting location tags
# this block searches first the title and then the text to extract a state or province and then the country

test_text = df.iat[4, 3]
test_loc = df.iat[4,1]

# United States
US_STATES_FULL = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
    "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
    "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New Hampshire","New Jersey","New Mexico","New York","North Carolina",
    "North Dakota","Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island",
    "South Carolina","South Dakota","Tennessee","Texas","Utah","Vermont",
    "Virginia","Washington","West Virginia","Wisconsin","Wyoming",
    "District of Columbia"
]

# Canada
CAN_PROVINCES_FULL = [
    "Alberta","British Columbia","Manitoba","New Brunswick",
    "Newfoundland and Labrador","Nova Scotia","Ontario",
    "Prince Edward Island","Quebec","Saskatchewan",
    "Northwest Territories","Nunavut","Yukon", "Baffin Island"
]

# Mexico
MEX_STATES_FULL = [
    "Aguascalientes","Baja California","Baja California Sur","Campeche","Chiapas",
    "Chihuahua","Coahuila","Colima","Durango","Guanajuato","Guerrero","Hidalgo",
    "Jalisco","Mexico City","México","Michoacán","Morelos","Nayarit",
    "Nuevo León","Oaxaca","Puebla","Querétaro","Quintana Roo","San Luis Potosí",
    "Sinaloa","Sonora","Tabasco","Tamaulipas","Tlaxcala","Veracruz",
    "Yucatán","Zacatecas"
]


# Lookup Dictionary
REGION_LOOKUP = {}

for state in US_STATES_FULL:
    REGION_LOOKUP[state.lower()] = ("US", state)

for prov in CAN_PROVINCES_FULL:
    REGION_LOOKUP[prov.lower()] = ("CA", prov)

for state in MEX_STATES_FULL:
    REGION_LOOKUP[state.lower()] = ("MX", state)

# Build regex pattern 
sorted_regions = sorted(REGION_LOOKUP.keys(), key=lambda x: -len(x))
pattern = r'\b(?:' + '|'.join(re.escape(name) for name in sorted_regions) + r')\b'
REGION_REGEX = re.compile(pattern, flags=re.IGNORECASE)


# Extraction Function
def extract_state_province(text: str):
    if not isinstance(text, str) or not text.strip():
        return None

    match = REGION_REGEX.search(text)

    if match:
        region = match.group(0).lower()
        return REGION_LOOKUP[region]

    return None


# Add the locations to our dataframe
df['Country'] = df['Accident Title'].apply(
    lambda x: extract_state_province(x)[0] 
    if extract_state_province(x) is not None 
    else None
)

df['State_Province'] = df['Accident Title'].apply(
    lambda x: extract_state_province(x)[1] 
    if extract_state_province(x) is not None 
    else None
)

# If the location was not in the title, try again in the text
mask = df['Country'].isna()
df.loc[mask, 'Country'] = df.loc[mask, 'Text'].apply(lambda x: extract_state_province(x)[0]if extract_state_province(x) is not None else np.nan)
mask = df['State_Province'].isna()
df.loc[mask, 'State_Province'] = df.loc[mask, 'Text'].apply(lambda x: extract_state_province(x)[1]if extract_state_province(x) is not None else np.nan)

print(df['Country'].value_counts())
print(df['State_Province'].value_counts())


Country
US    2231
CA     470
MX       2
Name: count, dtype: int64
State_Province
California               419
Colorado                 354
Alberta                  314
Alaska                   294
Washington               230
Wyoming                  186
Oregon                   137
British Columbia         114
New Hampshire            108
Utah                      86
North Carolina            79
New York                  47
West Virginia             42
Arizona                   37
Kentucky                  34
Nevada                    26
Idaho                     25
New Mexico                15
Quebec                    15
Wisconsin                 13
Maine                     12
Pennsylvania              11
Ontario                   11
Montana                   11
Yukon                     10
Virginia                   9
Tennessee                  8
Vermont                    6
Arkansas                   5
Illinois                   5
Texas                      5
Minnesota          

In [13]:
# more granular location tagging using mountainproject top 1000 most popular rock and ice routes
# want to get the most specific location for a route and not just the country/state it is in
# The mp data contains the route name, one or more aliases, lat, long, and the mp url

tqdm.pandas()
routes = pd.read_csv("routes_matcher_combined.csv")

# Helper functions
def norm(s):
    '''Normalize text for matching:
    - remove accents 
    - lowercase
    - remove whitespace
    '''
    if s is None or pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def canon_country(x):
    '''
    Standardize country names so both datasets align
    '''
    x = norm(x)
    if x in {"ca", "canada"}:
        return "Canada"
    if x in {"us", "usa", "united states", "united states of america"}:
        return "United States"
    if x in {"mx", "mexico"}:
        return "Mexico"
    return str(x).strip()

# Province / region / state name cleanup so the two CSVs line up better
PROVINCE_MAP = {
    "yukon": "Yukon Territory",
    "yukon territory": "Yukon Territory",
    "northwest territories": "Northwest Territories",
    "newfoundland": "Newfoundland and Labrador",
    "labrador": "Newfoundland and Labrador",
    "quebec": "Quebec",
    "british columbia": "British Columbia",
    "alberta": "Alberta",
    "ontario": "Ontario",
    "nova scotia": "Nova Scotia",
    "new brunswick": "New Brunswick",
    "manitoba": "Manitoba",
    "saskatchewan": "Saskatchewan",
    "nunavut": "Nunavut",
    "prince edward island": "Prince Edward Island",
    "baffin island": "Baffin Island",
    "new mexico": "New Mexico",
    "california": "California",
    "colorado": "Colorado",
    "washington": "Washington",
}

PROVINCE_MAP.update({
    "aguascalientes": "Aguascalientes",
    "baja california": "Baja California",
    "baja california sur": "Baja California Sur",
    "campeche": "Campeche",
    "chiapas": "Chiapas",
    "chihuahua": "Chihuahua",
    "coahuila": "Coahuila",
    "colima": "Colima",
    "durango": "Durango",
    "guanajuato": "Guanajuato",
    "guerrero": "Guerrero",
    "hidalgo": "Hidalgo",
    "jalisco": "Jalisco",
    "mexico city": "Mexico City",
    "méxico city": "Mexico City",
    "mexico state": "México",
    "méxico": "México",
    "michoacan": "Michoacán",
    "michoacán": "Michoacán",
    "morelos": "Morelos",
    "nayarit": "Nayarit",
    "nuevo leon": "Nuevo León",
    "nuevo león": "Nuevo León",
    "oaxaca": "Oaxaca",
    "puebla": "Puebla",
    "queretaro": "Querétaro",
    "querétaro": "Querétaro",
    "quintana roo": "Quintana Roo",
    "san luis potosi": "San Luis Potosí",
    "san luis potosí": "San Luis Potosí",
    "sinaloa": "Sinaloa",
    "sonora": "Sonora",
    "tabasco": "Tabasco",
    "tamaulipas": "Tamaulipas",
    "tlaxcala": "Tlaxcala",
    "veracruz": "Veracruz",
    "yucatan": "Yucatán",
    "yucatán": "Yucatán",
    "zacatecas": "Zacatecas",
})

PROVINCE_MAP.update({
    "al": "Alabama", "alabama": "Alabama",
    "ak": "Alaska", "alaska": "Alaska",
    "az": "Arizona", "arizona": "Arizona",
    "ar": "Arkansas", "arkansas": "Arkansas",
    "ca": "California", "california": "California",
    "co": "Colorado", "colorado": "Colorado",
    "ct": "Connecticut", "connecticut": "Connecticut",
    "de": "Delaware", "delaware": "Delaware",
    "fl": "Florida", "florida": "Florida",
    "ga": "Georgia", "georgia": "Georgia",
    "hi": "Hawaii", "hawaii": "Hawaii",
    "id": "Idaho", "idaho": "Idaho",
    "il": "Illinois", "illinois": "Illinois",
    "in": "Indiana", "indiana": "Indiana",
    "ia": "Iowa", "iowa": "Iowa",
    "ks": "Kansas", "kansas": "Kansas",
    "ky": "Kentucky", "kentucky": "Kentucky",
    "la": "Louisiana", "louisiana": "Louisiana",
    "me": "Maine", "maine": "Maine",
    "md": "Maryland", "maryland": "Maryland",
    "ma": "Massachusetts", "massachusetts": "Massachusetts",
    "mi": "Michigan", "michigan": "Michigan",
    "mn": "Minnesota", "minnesota": "Minnesota",
    "ms": "Mississippi", "mississippi": "Mississippi",
    "mo": "Missouri", "missouri": "Missouri",
    "mt": "Montana", "montana": "Montana",
    "ne": "Nebraska", "nebraska": "Nebraska",
    "nv": "Nevada", "nevada": "Nevada",
    "nh": "New Hampshire", "new hampshire": "New Hampshire",
    "nj": "New Jersey", "new jersey": "New Jersey",
    "nm": "New Mexico", "new mexico": "New Mexico",
    "ny": "New York", "new york": "New York",
    "nc": "North Carolina", "north carolina": "North Carolina",
    "nd": "North Dakota", "north dakota": "North Dakota",
    "oh": "Ohio", "ohio": "Ohio",
    "ok": "Oklahoma", "oklahoma": "Oklahoma",
    "or": "Oregon", "oregon": "Oregon",
    "pa": "Pennsylvania", "pennsylvania": "Pennsylvania",
    "ri": "Rhode Island", "rhode island": "Rhode Island",
    "sc": "South Carolina", "south carolina": "South Carolina",
    "sd": "South Dakota", "south dakota": "South Dakota",
    "tn": "Tennessee", "tennessee": "Tennessee",
    "tx": "Texas", "texas": "Texas",
    "ut": "Utah", "utah": "Utah",
    "vt": "Vermont", "vermont": "Vermont",
    "va": "Virginia", "virginia": "Virginia",
    "wa": "Washington", "washington": "Washington",
    "wv": "West Virginia", "west virginia": "West Virginia",
    "wi": "Wisconsin", "wisconsin": "Wisconsin",
    "wy": "Wyoming", "wyoming": "Wyoming",
})

def canon_province(x):
    '''
    Normalize province/state names using lookup map
    '''
    x_norm = norm(x)
    return PROVINCE_MAP.get(x_norm, str(x).strip())

# Generic alias blacklist - do not want these listed as the precise location
COUNTRY_NAMES = {
    "canada",
    "united states",
    "united states of america",
    "usa",
    "us",
    "mexico",
    "mx",
}

CANADA_PROVINCES = {
    "alberta",
    "british columbia",
    "manitoba",
    "new brunswick",
    "newfoundland and labrador",
    "newfoundland",
    "labrador",
    "nova scotia",
    "ontario",
    "prince edward island",
    "quebec",
    "saskatchewan",
    "northwest territories",
    "nunavut",
    "yukon",
    "yukon territory",
}

US_STATES = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "district of columbia", "florida", "georgia",
    "hawaii", "idaho", "illinois", "indiana", "iowa", "kansas", "kentucky",
    "louisiana", "maine", "maryland", "massachusetts", "michigan",
    "minnesota", "mississippi", "missouri", "montana", "nebraska", "nevada",
    "new hampshire", "new jersey", "new mexico", "new york", "north carolina",
    "north dakota", "ohio", "oklahoma", "oregon", "pennsylvania",
    "rhode island", "south carolina", "south dakota", "tennessee", "texas",
    "utah", "vermont", "virginia", "washington", "west virginia", "wisconsin",
    "wyoming"
}

MEXICO_STATES = {
    "aguascalientes", "baja california", "baja california sur", "campeche",
    "chiapas", "chihuahua", "coahuila", "colima", "durango", "guanajuato",
    "guerrero", "hidalgo", "jalisco", "mexico city", "méxico city", "michoacan",
    "michoacán", "morelos", "nayarit", "nuevo leon", "nuevo león", "oaxaca",
    "puebla", "queretaro", "querétaro", "quintana roo", "san luis potosi",
    "san luis potosí", "sinaloa", "sonora", "tabasco", "tamaulipas",
    "tlaxcala", "veracruz", "yucatan", "yucatán", "zacatecas", "méxico", "mexico"
}

GENERIC_HIERARCHY_TERMS = {
    "north america",
    "international",
    "area",
    "region",
    "province",
    "state",
    "park",
    "national park",
    "wilderness",
    "forest",
}


# Combine all  generic terms
GENERIC_ALIAS_TERMS = {
    *COUNTRY_NAMES,
    *CANADA_PROVINCES,
    *US_STATES,
    *MEXICO_STATES,
    *GENERIC_HIERARCHY_TERMS,
}

def is_generic_alias(term):
    '''
    Returns True if a term is too generic to be useful for matching
    '''
    return norm(term) in GENERIC_ALIAS_TERMS

# Standardize both dataframes the same way
df["Country_std"] = df["Country"].apply(canon_country)
df["State_Province_std"] = df["State_Province"].apply(canon_province)

routes["country_std"] = routes["country"].apply(canon_country)
routes["province_std"] = routes["province"].apply(canon_province)

# Route aliases into lists
def split_aliases(x):
    '''
    Convert alias string into list:
    "El Cap | The Captain" -> ["El Cap", "The Captain"]
    '''
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [a.strip() for a in str(x).split("|") if a.strip()]

routes["alias_list"] = routes["aliases"].apply(split_aliases)

# Build province-specific lookup
# key = (country_std, province_std)
# value = { normalized term: info }

def build_region_lookup(routes_df):
    region_lookup = defaultdict(list)

    for _, row in routes_df.iterrows():
        key = (row["country_std"], row["province_std"])

        info = {
            "matched_name": row["name"],
            "country": row["country_std"],
            "province": row["province_std"],
            "lat": row["lat"],
            "lon": row["lon"],
            "route": row.get("Route", None),
            "url": row.get("URL", None),
        }

        ordered_terms = [row["name"]] + list(row["alias_list"])

        for term in ordered_terms:
            term_norm = norm(term)
            if not term_norm:
                continue

            # keep the canonical name even if it looks generic,
            # but filter generic aliases
            if term_norm != norm(row["name"]) and is_generic_alias(term_norm):
                continue

            region_lookup[key].append({
                "term": term_norm,
                "info": info
            })

    return region_lookup

region_lookup = build_region_lookup(routes)


# Exact matcher for one row
def extract_location_for_row(text, country, province, region_lookup):
    '''
    Try to find a route/location name inside accident text.
    Matching is:
    - restricted to correct country/province
    - exact word match 
    '''
    if not isinstance(text, str) or not text.strip():
        return None

    key = (canon_country(country), canon_province(province))
    entries = region_lookup.get(key)

    if not entries:
        return None

    text_norm = norm(text)

    for entry in entries:
        term = entry["term"]
        pattern = r"(?<!\w)" + re.escape(term) + r"(?!\w)"
        if re.search(pattern, text_norm):
            result = entry["info"].copy()
            result["matched_phrase"] = term
            result["score"] = 100
            return result

    return None

def extract_location_fields(row):
    '''
    - runs matcher
    - attaches aliases 
    - formats output fields
    '''
    result = extract_location_for_row(
        row["Accident Title"],
        row["Country"],
        row["State_Province"],
        region_lookup
    )

    if result:
        matched = routes[
            (routes["name"] == result["matched_name"]) &
            (routes["country_std"] == canon_country(row["Country"])) &
            (routes["province_std"] == canon_province(row["State_Province"]))
        ].head(1)

        aliases = ""
        if len(matched) > 0:
            aliases = matched.iloc[0]["aliases"]
            if pd.isna(aliases):
                aliases = ""

        display_location = result["matched_name"]
        if aliases:
            display_location = f"{result['matched_name']} | aliases: {aliases}"

        return display_location, result["matched_phrase"], result["lat"], result["lon"]

    return None, None, None, None


df[["location_name", "matched_phrase", "lat", "lon"]] = df.progress_apply(
    extract_location_fields,
    axis=1,
    result_type="expand"
)

print(df['location_name'].value_counts())
print(df['matched_phrase'].value_counts())

100%|██████████| 2770/2770 [00:19<00:00, 142.30it/s]

location_name
2. Southwest Face | aliases: B. El Capitan|Valley North Side|Yosemite Valley|Yosemite National Park|California                              232
Mount Moran | aliases: Grand Teton National Park|Wyoming                                                                                    113
Redgarden - Roof Routes | aliases: Redgarden Wall|Eldorado Canyon State Park|Boulder|Colorado                                               107
Mount Rainier | aliases: Mount Rainier National Park|Southwest Cascades|South-West & Tacoma|Washington                                       85
Mt. Temple | aliases: Valley of the Ten Peaks|Banff National Park|Alberta|North America|International                                        76
                                                                                                                                           ... 
East Rosebud Ice Climbing | aliases: East Rosebud|Beartooth Mountains|South Central Region|Montana                        

In [14]:
"""
Tagging data based on type of climbing, helmet, rope:
    - Type: Rock, Ice, Alpine
    - Roped or Unroped
    - Helmet or no helmet
    
"""

#climbing tags
climb_tag = {
        "alpine": ["alpine", 
                   "alpinists", 
                   "mountaineering", 
                   "mountaineers", 
                   "glissade", 
                   "summit", 
                   "face", 
                   "ridge", 
                   "gully", 
                   "couloir", 
                   "glacier", 
                   "cornice", 
                   "corniced", 
                   "crevasse", 
                   "col", 
                   "bergschrund", 
                   "third class", 
                   "fourth class", 
                   "class-four", 
                   "blade serac", 
                   "pinnacle", 
                   "summiting", 
                   "mountain", 
                   "peak", 
                   "Red Banks",
                   "avalanche",
                   "mixed",
                   "McKinley",
                   "Fuhrer's Finger",
                   "Mount Shuksan",
                   ],
        "rock": ["crux", 
                 "slab", 
                 "slabs", 
                 "piton", 
                 "top rope", 
                 "top-rope", 
                 "Friend", 
                 "chock", 
                 "5.", 
                 "spire", 
                 "trad", 
                 "pillar", 
                 "rock climbing", 
                 "Trapps", 
                 "Robinson Park", 
                 "East Ledges", 
                 "Oak Creek Vista", 
                 "free climb", 
                 "bouldering", 
                 "boulder problem", 
                 "sport route", 
                 "quickdraws", 
                 "bolt", 
                 "Jill's Thrill", 
                 "bolts", 
                 "Empor", 
                 "Foothill Crag", 
                 "Church Vaults", 
                 "Coal Pit Gulch", 
                 "Blockbuster", 
                 "Three Pines", 
                 "Bruise Brothers wall", 
                 "sport climb", 
                 "Jolly Rodger", 
                 "top- roping", 
                 "sport climbing", 
                 "Directissima Route", 
                 "Hemateria", 
                 "East Buttress route", 
                 "Dire Straights", 
                 "The Bastille", 
                 "top-roping", 
                 "Parleys Canyon", 
                 "Chimney Pond", 
                 "Bastille Crack", 
                 "Rincon Wall", 
                 "Birdland", 
                 "Grasshopper", 
                 "Vertical Endeavors Climbing Gym", 
                 "Whitney Gilman rock climb", 
                 "Fairview Dome", 
                 "Solid Gold", 
                 "China Wall", 
                 "Condor Crag", 
                 "Commitment", 
                 "Boulderado Crag", 
                 "Delay of the Game", 
                 "Poudre Canyon", 
                 "Flatiron", 
                 "Happy Hour Crag",
                 "Hidden Valley",
                 "Mount Brock",
                 "Kid Goat",
                 "Tohasket",
                 "Sunnyside",
                 "Devil's Lake",
                 "Snaz",
                 "Royal Arches",
                 "Smuggler's Notch",
                 "Old Man’s Route",
                 "Chapel Ledges",
                 "Open Book",
                 "Spaceshot",
                 "Windy Point",
                 "Recompense",
                 "The Nose",
                 "Feast of Snakes",
                 "Solarium",
                 "Thatcher Park",
                 "Mickey’s Beach",
                 "Sycamore Falls",
                 "Lemon Reservoir",
                 "Sturs Chimney",
                 "Star Trek Wall",
                 "Frothing Green",
                 "Cooper’s Rock",
                 "New River Gorge"
                 ],
        "ice": ["ice climbers", 
                "ice-climbers", 
                "smear", 
                "ice-capped", 
                "crampon", 
                "ice climb", 
                "ice route", 
                "WI4 route", 
                "WI4", 
                "Ouray Ice Park", 
                "Buttermilk Falls",
                "Pearl Necklace",
                ]}

#roping tags
roped = ["belay", 
         "belayers", 
         "climbing rope", 
         "figure 8", 
         "figure-8", 
         "roped", 
         "ropelengths", 
         "rappel", 
         "rappelled", 
         "rappelling", 
         "rope team", 
         "fixed line", 
         "tied in", 
         "Grade 3", ]
unroped = ["unroped",
           "scramble", 
           "scrambler", 
           "solo", 
           "downclimbing",
           "no rope"]

#helmet tags NEEDS UPDATINF
helmet = ["wearing helmet", 
          "wore helmet", 
          "with helmet", 
          "helmeted"
          "wear a helmet",
          "wearing a helmet",
          "had a helmet on",
          "helmet was on",
          "put on helmet",
          "put on a helmet",
          "climbing helmet",
          "his helmet",
          "her helmet",
          "their helmet",
          "helmet proved",
          "helmet saved",
          "helmet prevented",
          "helmet took",
          "helmet absorbed",
          "helmet protected"]
no_helmet = ["no helmet", 
             "not wearing a helmet",
             "helmet recommended"
             "not wearing helmet",
             "without a helmet",
             "without helmet",
             "no helmets",
             "neither climber was wearing a helmet",
             "neither was wearing a helmet",
             "was not wearing a helmet",
             "were not wearing helmets",
             "had removed his helmet",
             "had removed her helmet",
             "helmet was removed",
             "took off helmet",
             "removed helmet"]
    


#fucntions for tags
def safe_text(text):
    if pd.isna(text):
        return ""
    
    text = text.lower()
    text = text.replace("-", " ")
    return text

def tag_climb(text):
    text = safe_text(text)
    if not text:
        return []
    
    tags = []
    for tag, keywords in climb_tag.items():
        for k in keywords:
            if k.lower() in text:
                tags.append(tag)
                break

    return tags

priority = ["ice", "rock", "alpine"]

def tag_primary_climb(text):
    tags = tag_climb(text)
    for p in priority:
        if p in tags:
            return p
    return "unknown"

def tag_rope(text):
    text = safe_text(text)

    rope_score = 0
    no_rope_score = 0

    if re.search(r"(unknown|unclear|not known).{0,10}(rope|belay)", text):
        return "unknown"

    if re.search(r"(solo|free solo|soloing|unroped|without rope)", text):
        no_rope_score += 2

    if re.search(r"(no belay|not belayed)", text):
        no_rope_score += 2

    if re.search(r"(scramble|scrambling)", text):
        no_rope_score += 1

    if re.search(r"(belay|belayed|on belay)", text):
        rope_score += 2

    if re.search(r"(rappel|rappelling|rappelled)", text):
        rope_score += 2

    if re.search(r"(lowered|lowering)", text):
        rope_score += 2

    if re.search(r"(tied in|rope team|fixed line)", text):
        rope_score += 2

    if re.search(r"(leader|leading|lead climber)", text):
        rope_score += 2

    if re.search(r"(second|seconding|follower)", text):
        rope_score += 2

    if re.search(r"(pitch|multi pitch|pitch)", text):
        rope_score += 1

    if re.search(r"(caught|held).{0,10}rope", text):
        rope_score += 2

    if rope_score > no_rope_score and rope_score >= 1.5:
        return "rope"
    if no_rope_score > rope_score and no_rope_score >= 1.5:
        return "no_rope"

    return "unknown"


def tag_helmet(text):
    text = safe_text(text)

    if re.search(r"(unknown|unclear|not known).{0,10}helmet", text):
        return "unknown"

    if re.search(r"(might|may|could|would).{0,15}helmet", text):
        return "unknown"

    if re.search(r"(no|not|without|lacked|unhelmeted|bareheaded).{0,15}(helmet|headgear|protection)", text):
        return "no_helmet"

    if re.search(r"(no headgear|no helmet|no protective headgear|without head protection)", text):
        return "no_helmet"

    if re.search(r"(wearing|wore|had|with|helmeted|put on).{0,10}helmet", text):
        return "helmet"

    if re.search(r"(his|her|their).{0,3}helmet", text):
        return "helmet"

    if re.search(r"(helmet).{0,10}(struck|hit|impacted|cracked|damaged|absorbed)", text):
        return "helmet"

    if re.search(r"(landed on|fell on).{0,10}helmet", text):
        return "helmet"

    if re.search(r"(helmet).{0,10}(came off|knocked off|lost|dislodged)", text):
        return "helmet"


    if any(k in text for k in no_helmet):
        return "no_helmet"
    
    if any(k in text for k in helmet):
        return "helmet"

    return "unknown"


df.columns = df.columns.str.strip()
df['climb_tags'] = df['Text'].apply(tag_climb)

def primary_from_tags(tags):
    for p in priority:
        if p in tags:
            return p
    return "unknown"

df['climb_primary'] = df['climb_tags'].apply(primary_from_tags)

df['Helmet'] = df['Text'].apply(tag_helmet)
df['Rope'] = df['Text'].apply(tag_rope)

print(df['climb_primary'].value_counts())
print(df['Helmet'].value_counts())
print(df['Rope'].value_counts())
print (df.head())

climb_primary
rock       1494
alpine      735
ice         469
unknown      72
Name: count, dtype: int64
Helmet
unknown      2152
helmet        336
no_helmet     282
Name: count, dtype: int64
Rope
rope       1935
unknown     664
no_rope     171
Name: count, dtype: int64
  ID                                     Accident Title  Publication Year  \
0  1  Failure of Rappel Setup (Protection Pulled Out...            1990.0   
1  2  Failure of Rappel—Failure to Check System, Bri...            1990.0   
2  3  Fall into Crevasse, Climbing Alone, Inadequate...            1990.0   
3  4  Fall into Crevasse, Climbing Unroped, British ...            1990.0   
4  5  Fall Into Crevasse, Unroped, Inadequate Equipm...            1990.0   

                                                Text  \
0  Colorado, Rocky Mountain National Park\nOn May...   
1  British Columbia, Squamish, Smoke Bluffs\nOn M...   
2  Alberta, Rocky Mountains, Crowfoot Mountain\nO...   
3  British Columbia, Bugaboo Mountains, Bug

In [ ]:
df.to_csv('ANAC_textmatch.csv', index=False)